# GridDividend
## A Model for Electric Utility Reform Through Shared Savings

**Dr Robert A. F. Currie, May 2026**

Companion model to: *GridDividend: A Model for Electric Utility Reform* (Substack, 2026)  
Based on: *Electric Utility Reform in an Age of Electrification* (The States Forum, 2025)  
Repository: github.com/robertafcurrie/GridDividend

---

The United States is at an inflection point in how it powers itself. Electricity demand is rising for the first time in decades, driven by electric vehicles, heat pumps, data centres, and industrial electrification. The grid that will carry this load is being built now — in rate cases, capital programmes, and regulatory decisions made state by state, utility by utility, year by year. The choices made in the next decade will compound for half a century. This is not primarily a crisis. It is an opportunity — one of the largest in the history of the American electric industry. States and utilities that move early to align financial incentives with efficient outcomes will build a cleaner, more resilient, and more affordable grid than the one the current regulatory model would produce. The window for states AND utilities to lead on this is open. GridDividend is a tool for exploring what leadership could look like in practice — not a finished answer, but a structured starting point for the conversations that need to happen.

### What this model is — and is not

GridDividend is a **strategic policy simulation framework**. It is designed to show the *direction and order of magnitude* of the shared savings opportunity using real regulatory starting parameters and transparent assumptions. It is **not** a utility planning model, a rate case filing, or a forecast.

Three specific scope boundaries matter most:

- **Full tariff bill, not all bill components.** The bill figures include both delivery (distribution infrastructure) and supply (energy and NYISO capacity costs), anchored to filed tariff rates for a 600 kWh/month customer. Shared savings reform acts primarily on the delivery component through infrastructure avoidance, and secondarily on supply through lower peak demand reducing NYISO capacity obligations. What the model does not project are independent movements in wholesale energy prices, transmission costs, or policy surcharges — these are held at their starting growth trajectory. Reform savings represent what a distribution utility under shared savings incentives can influence, not a forecast of total electricity bill movements.
- **Spatially aggregate.** The model treats the grid as a system-wide pool of capacity. Real distribution systems are constrained networks where NWS value depends on *where* the DER is relative to *where* the constraint is. Individual feeders will see very different outcomes from the system average.
- **One representative scenario.** Results are a single deterministic trajectory, not a probability distribution. Many parameters are uncertain and correlated in ways the model does not capture.
- **AI assistance.** AI was used extensively in researching and creating this model, and in writing the companion article.

These limitations are discussed fully in Section 10.

### How to use this notebook

1. **Edit Section 1** to change any parameter
2. **Run All Cells** (Kernel → Restart & Run All)
3. Charts save to `charts/` folder

All bill figures are for a 600 kWh/month residential customer — the standard PSC benchmark.  
Results cover 2026–2050. BAU = Business As Usual. SS = Shared Savings Reform.

---

## Section 0: Setup
Imports, styling, and output directory.

In [ ]:
# ============================================================
# SECTION 0: SETUP
# Self-contained — no external files required.
# All utility data is embedded below.
# Requires: pandas, numpy, matplotlib (all standard Anaconda/pip packages)
# Install if needed: pip install pandas numpy matplotlib
# ============================================================

import sys, os
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.35,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'figure.dpi': 130,
})
C = {
    'bau': '#d62728', 'shared': '#2ca02c',
    'freeze_bau': '#ff7f0e', 'freeze_shared': '#17becf',
    'util_bau': '#ff7f0e', 'util_shared': '#1f77b4',
    'neutral': '#888888', 'tax': '#9467bd',
    'capex_spent': '#1f77b4', 'capex_avoided': '#2ca02c',
    'opex_nws': '#ff7f0e', 'ineligible': '#c5b0d5',
}
MY = {2030: 'CLCPA\n70% Renew.', 2035: 'Peak\nElectrif.', 2040: 'CLCPA\nNet-Zero'}
YEARS = list(range(2026, 2051))
os.makedirs('charts', exist_ok=True)

# ── Utility data (embedded — no external file needed) ─────────────────────────
#
# Sources:
#   Con Edison: NYPSC Case 25-E-0072, Joint Proposal adopted January 22, 2026
#               Con Edison 2024 Form 10-K; SEC 8-K November 2025
#   RG&E:       NYPSC Case 25-E-0379 — temporary rates set June 1 2026 (+4.0% electric)
#               Final PSC order still pending. Parameters estimated from June 2025 filing.
#               ⚠ RG&E rate base is back-calculated; update when PSC order issues.

UTILITIES = {
    "ConEd": {
        "name": "Consolidated Edison (CECONY)",
        "short_name": "Con Edison",
        "state": "New York",
        "territory": "New York City and Westchester County",
        "filing_case": "Case 25-E-0072",
        # Rate base & capital (Source: Joint Proposal, approved Jan 2026)
        "rate_base_start_m": 32935,       # $M electric rate base, 2026 average
        "annual_capex_m": 4579,           # $M/yr (RY1=$4,550M RY2=$4,474M RY3=$4,712M average)
        "total_revenue_m": 8400,          # $M total electric revenue requirement (estimated)
        # Capital structure (Source: Joint Proposal 2026-2028)
        "roe": 0.094,                     # Allowed ROE 9.4%
        "equity_ratio": 0.48,             # Equity share of capital structure
        # Customers & load (Source: Con Edison 2024 10-K)
        "electric_customers": 3_700_000,
        "peak_demand_mw": 11822,          # Summer 2024 peak
        "annual_energy_gwh": 63000,
        # Data center load
        "dc_load_growth_annual": 0.008,   # 0.8%/yr additional load from data centres
        "dc_distribution_share": 0.15,    # 15% of new DC load at distribution level
        "dc_flexibility_share": 0.25,     # 25% of DC load enrollable in demand response
        # DER baseline 2026 (Source: CLCPA progress reports; NYSERDA data)
        "der_storage_mw_start": 150,
        "ev_managed_charging_pct_bau": 0.10,
    },
    "RGE": {
        "name": "Rochester Gas and Electric Corporation",
        "short_name": "RG&E",
        "state": "New York",
        "territory": "Nine-county region centred on Rochester, NY",
        "filing_case": "Case 25-E-0379 (temporary rates Jun 2026; final order pending)",
        # Rate base & capital — ESTIMATED from June 2025 filing.
        # PSC set temporary rates June 1 2026 (+4.0% electric); final order still pending.
        # Update these parameters when the final order is issued.
        "rate_base_start_m": 1600,        # $M — back-calculated estimate ⚠
        "annual_capex_m": 235,            # $M/yr — derived from filing capital driver
        "total_revenue_m": 1096,          # $M — derived from filing
        # Capital structure (Source: Case 25-E-0379 filing)
        "roe": 0.100,                     # Requested ROE 10.0%
        "equity_ratio": 0.48,
        # Customers & load
        "electric_customers": 383_000,
        "peak_demand_mw": 1050,
        "annual_energy_gwh": 5500,
        # Data center load (higher than ConEd — upstate NY actively targeted)
        "dc_load_growth_annual": 0.018,   # 1.8%/yr
        "dc_distribution_share": 0.25,
        "dc_flexibility_share": 0.35,
        # DER baseline 2026
        "der_storage_mw_start": 60,
        "ev_managed_charging_pct_bau": 0.08,
    }
}

# Shared parameters applying to both utilities
SHARED_DEFAULTS = {
    "ev_adoption_curve": {   # Statewide NY EVs (CLCPA-anchored, NYSERDA targets)
        2026: 1_000_000,
        2030: 3_000_000,
        2035: 6_000_000,
        2040: 8_500_000,
        2050: 10_000_000,
    },
    "ev_peak_kw": 7.2,       # Average peak draw per EV (Level 2 charging, kW)
}

print("GridDividend v12 — self-contained notebook, no external files needed.")
print(f"Modelling years: {YEARS[0]}–{YEARS[-1]}")
print(f"Charts → charts/")
print()
print("Utilities loaded:")
for k, u in UTILITIES.items():
    print(f"  {u['short_name']} ({u['filing_case']})")

## Section 1: Parameters

**Edit any value here. Then re-run all cells.**

All parameters are documented with their sources and rationale.
See the companion article for full discussion of each assumption.

In [ ]:
# ============================================================
# SECTION 1: USER-ADJUSTABLE PARAMETERS
# ============================================================

# ── Utilities to model ────────────────────────────────────────────────────────
utilities_to_run = ['ConEd', 'RGE']   # 'ConEd', 'RGE', or both

# ── Shared savings mechanism ──────────────────────────────────────────────────
UTILITY_SAVINGS_SHARE = 0.30
# Fraction of savings pool retained by utility. Default 30%.
# Based on NY PSC precedent for NWS incentive mechanisms.
# Remainder (70%) is returned to customers as bill savings.

# ── NWS-eligible CAPEX ────────────────────────────────────────────────────────
NWS_ELIGIBLE_CAPEX_SHARE_START = 0.30
# Starting fraction of annual CAPEX addressable by NWS (~30%).
# Consistent with proportion of Con Ed rate case increase driven by
# new infrastructure investment. Brattle Ontario study found 20-40%
# growth capex reduction at provincial scale — consistent with this range.

NWS_ELIGIBLE_CAPEX_SHARE_MAX = 0.50
# Maximum eligible share as electrification drives more load-growth CAPEX.
# Ceiling reflects that asset replacement never disappears from the programme.

NWS_ELIGIBLE_CAPEX_LOAD_SENSITIVITY = 2.0
# Rate at which eligible share grows with cumulative electrification load.

NWS_AVOIDANCE_OF_ELIGIBLE_START = 0.20
# Fraction of eligible CAPEX NWS defers in year 1.
# Low in early years: programmes nascent, DER aggregation immature.

NWS_AVOIDANCE_OF_ELIGIBLE_MATURE = 0.80
# Fraction of eligible CAPEX NWS defers at programme maturity.
# Supported by two bodies of evidence:
# (1) Brattle Ontario (2026): NWS cost-effective in 95% of feeder scenarios
# (2) GB flexibility markets (UKPN DSO, 2025): 31 GW tendered, 9 GW contracted,
#     mature DSO programmes meeting 100% of identified network constraints at
#     specific constrained locations, delivering £300M consumer savings in 2024.
# Source: UKPN DSO Performance Panel Report 2024/25; Blake Clough Consulting
# (2025), DSO Flexibility Markets: Transforming GB's Grid Economics.
# The 80% reflects a mature programme; 20% residual covers upgrades too large,
# too remote, or insufficiently served by available DER to be deferred by NWS.

NWS_AVOIDANCE_RAMP_YEARS = 9
# Years to ramp from START to MATURE avoidance rate.

# ── Utilisation credit — Brattle Untapped Grid mechanism ─────────────────────
FLEX_LOAD_UTILISATION_SHARE_SS = 0.15
# Fraction of annual load growth served from existing system headroom (SS only).
#
# Mechanism: flexible load (managed EVs, curtailable data centres, demand
# response) uses spare capacity without triggering proportional CAPEX growth.
# Fixed costs are spread over more kWh → delivery rate per kWh falls.
# This is independent of and additive to NWS CAPEX avoidance.
#
# Source: Brattle/Utilize Coalition (2026), "The Untapped Grid":
# 10% utilisation improvement → 4.8% all-in rate reduction; utility earnings
# still grow +23%; new load connects 4-9 years faster.
# Default 15% is conservative vs Brattle base case (~33%) — appropriate
# because GridDividend models distribution scope only.
#
# Set to 0.0 to disable and isolate the other two mechanisms.

FLEX_LOAD_UTILISATION_SHARE_BAU = 0.00
# No intentional utilisation optimisation in BAU.

# ── NWS programme costs ───────────────────────────────────────────────────────
OPEX_RATIO = 0.15
# NWS OPEX as fraction of avoided CAPEX (default 15%).
# Utility pays DER aggregators, demand response providers, managed charger fees.
#
# DERIVATION: Brattle Ontario (2026) used $50/kW-year for DER capacity costs.
# At $2M/MW avoided upgrade cost and ~25% dispatch efficiency, avoided CAPEX of
# $1M corresponds to roughly 0.5 MW of NWS capacity at ~$25k/yr OPEX → ~2.5%.
# However, real NWS programmes also include programme administration, DERMS
# operation, customer acquisition, and performance monitoring — these
# substantially increase total OPEX. BQDM actual OPEX ran ~12-18% of avoided
# capital on an annualised basis (source: NY PSC BQDM programme reports).
#
# RANGE: 10-25% is defensible; 15% is the central estimate.
# This parameter has significant uncertainty and should be treated as a
# sensitivity variable. See Section 8 for sensitivity analysis.
# Future versions should model OPEX explicitly by contract type:
#   - Availability payments (paid regardless of dispatch)
#   - Utilisation payments (per MWh dispatched)
#   - Capacity payments (per kW committed)
# These have very different cost profiles and risk allocations.

NWS_DISPATCH_EFFICIENCY = 0.25
# Fraction of total flex stack reliably dispatchable as NWS at any given time.

# ── Distribution upgrade cost — set per utility ──────────────────────────────
# $M per MW of NWS-eligible distribution upgrade deferred.
# This parameter does significant work in the physical NWS capability check:
#   nws_dollar_value_m = nws_capable_mw × DIST_UPGRADE_COST_PER_MW
# It must reflect the utility's actual distribution capital cost structure.
#
# RANGE AND SOURCES:
# Con Edison (dense urban, almost entirely underground):
#   $7.1M/MW — Idlewild Project: $1.2B for 170 MW of load transfer capacity
#     (Source: NYPSC Case 23-E-0360, approved Jan 2024; Con Edison 10-K 2024)
#     Note: this is a major substation project. Routine feeder reinforcements
#     in Con Edison's underground network typically run $3-5M/MW.
#     Conservative central estimate: $4.0M/MW for the eligible CAPEX mix
#     (weighted average of substation projects and feeder reinforcements).
#
# RG&E (suburban/rural, predominantly overhead):
#   Overhead distribution reinforcement: $0.5-1.5M/MW
#   Substation upgrades: $1.5-3.0M/MW
#   Conservative central estimate: $1.5M/MW for the eligible CAPEX mix
#   (RG&E capital programme is primarily overhead suburban work at lower cost)
#
# IMPORTANT MODELLING NOTE:
# This parameter varies enormously — even within a single utility's programme.
# A routine overhead line upgrade might cost $0.3M/MW; an urban underground
# substation might cost $8-10M/MW. Using a single blended figure is a
# necessary simplification for this model. State-level analysis should:
#   (1) Obtain the utility's CAPEX programme by project type and voltage class
#   (2) Apply category-specific costs to the NWS-eligible subset
#   (3) Model the distribution of upgrade sizes rather than a blended average
# See Part Nine of the companion article for discussion.
#
# In the current model, the physical NWS capability check (nws_dollar_value_m)
# is typically NOT the binding constraint — the policy ceiling on eligible
# CAPEX avoidance binds first. The cost parameter therefore has limited
# impact on results in the base case but becomes important in scenarios
# with high DER availability or high avoidance rates.

DIST_UPGRADE_COST_PER_MW = {
    'ConEd': 4.0,   # $M/MW — urban underground mix (see documentation above)
    'RGE':   1.5,   # $M/MW — suburban/rural overhead mix
}
# To use a single value for all utilities: set as a number, e.g. DIST_UPGRADE_COST_PER_MW = 2.0

# ── Pass-through savings ──────────────────────────────────────────────────────
# ── Capacity market pass-through savings ─────────────────────────────────────
#
# Pass-through savings are computed as a post-processing step in Section 4:
#   saving_year_N = (peak_bau_N − peak_ss_N) MW × capacity_cost_year_N
#
# This uses the actual peak demand difference between BAU and SS scenarios
# rather than approximating from a fixed percentage of supply costs.
#
# CAPACITY_COST_START: NYISO ICAP clearing price per MW-year by load zone
#   ConEd: NYC zone — avg ~$120/kW-yr = $0.120M/MW-yr (NYISO 2024 auctions)
#   RGE:   Upstate zones G/H/I — avg ~$70/kW-yr = $0.070M/MW-yr
#   Includes both generation ICAP and transmission capacity components.
#   Sources: NYISO Installed Capacity Market reports 2024; NYISO ICAP auctions.
#
# CAPACITY_COST_GROWTH: Real annual growth in capacity prices (3%/yr)
#   Electrification and data centre load tighten reserve margins → higher prices.
#   New peaking capacity (gas CT or large-scale BESS) sets price ceiling.
#   Nominal growth ~5%/yr including inflation; 3% real is conservative.

CAPACITY_COST_START = {
    'ConEd': 0.120,   # $M/MW-year (NYC zone, 2026 anchor)
    'RGE':   0.070,   # $M/MW-year (upstate zones G/H/I, 2026 anchor)
}
CAPACITY_COST_GROWTH = 0.03   # 3%/yr real growth in capacity prices

# ── Load growth ───────────────────────────────────────────────────────────────
BASELINE_LOAD_GROWTH = 0.005   # 0.5%/yr baseline organic growth

ELECTRIFICATION_RAMP = {
    (2026, 2030): 0.008,   # EVs and heat pumps ramping: +0.8%/yr
    (2031, 2035): 0.018,   # Peak CLCPA electrification decade: +1.8%/yr
    (2036, 2040): 0.012,   # Post-peak: +1.2%/yr
    (2041, 2050): 0.005,   # Mature electrification: +0.5%/yr
}

# ── DER adoption ──────────────────────────────────────────────────────────────
STORAGE_MAX_PCT_OF_PEAK = 0.35   # Storage ceiling: 35% of peak (CLCPA-anchored)
STORAGE_GROWTH_BAU      = 0.15   # 15%/yr storage growth (BAU, market-driven)
STORAGE_GROWTH_SHARED   = 0.22   # 22%/yr under SS (utility actively promotes)
DER_ADOPTION_LEAD_YEARS = 4      # SS achieves same DER level 4 years ahead of BAU
EV_MANAGED_CHARGING_SHARED = 0.45   # 45% managed EV charging under SS (vs ~10% BAU)
STORAGE_PEAK_OFFSET_KW_PER_MWH = 0.85   # kW of peak offset per MWh storage capacity

# ── Financial parameters ──────────────────────────────────────────────────────
COST_OF_DEBT   = 0.05    # 5% cost of debt
INFLATION_RATE = 0.025   # 2.5%/yr general inflation

# ── Floor constraint ──────────────────────────────────────────────────────────
CAPEX_SPENT_FLOOR_GROWTH = 0.005
# Minimum annual growth in CAPEX spent (0.5%/yr).
# NWS defers FUTURE projects, not current year commitments already underway.
# Without this floor, the model could show CAPEX spending falling in nominal
# terms, which is not realistic for a utility with multi-year capital programmes.

# ── Rate freeze overlay ───────────────────────────────────────────────────────
RATE_FREEZE_YEARS          = 2   # Duration of rate freeze (0 to disable)
RATE_FREEZE_RECOVERY_YEARS = 4   # Catch-up period to recover deferred costs

# ── Residential bill anchor ───────────────────────────────────────────────────
RESIDENTIAL_KWH_MONTH = 600
# Standard PSC benchmark: 600 kWh/month residential customer.
# Used in all NY utility rate cases for illustrative bill comparisons.
# Actual usage: ~400 kWh/mo (small city apartment) to 1,000+ kWh/mo (suburban).
# Percentage savings apply regardless of usage level — reform acts on ¢/kWh rate.

UTILITY_BILL_PARAMS = {
    'ConEd': {
        'delivery_rate_kwh':  0.141,  # 14.1¢/kWh delivery (Case 25-E-0072, 2024 tariff)
        'supply_rate_kwh':    0.103,  # 10.3¢/kWh supply (2024 market average)
        'basic_charge_month': 18.0,   # $18/month basic service charge
        'delivery_share_rr':  0.60,   # ~60% of RR is delivery (vs supply pass-through)
        # Source: Con Edison SC-1 residential service tariff filed with NYPSC
    },
    'RGE': {
        'delivery_rate_kwh':  0.0741, # 7.41¢/kWh delivery (July 2024 tariff)
        'supply_rate_kwh':    0.080,  # 8.0¢/kWh supply (estimated market rate)
        'basic_charge_month': 27.0,   # $27/month basic service charge
        'delivery_share_rr':  0.56,   # ~56% of RR is delivery
        # Source: RG&E electric rate summary July 2024; supply rate estimated
        # ⚠️ RG&E Case 25-E-0379: PSC set temporary rates June 1 2026 (+4.0% electric).
        #    Final order pending. All RG&E figures estimated from June 2025 filing.
    },
}

# ── IMPORTANT NOTE ON PROPERTY TAX ───────────────────────────────────────────
# Property tax is set in load_utility() as a fraction of rate base.
# DEFAULT VALUES (electric assets only, approximate):
#   Con Edison: 1.6% of electric rate base (~$520M/yr on $32.9B rate base)
#   RG&E:       1.8% of electric rate base (~$29M/yr on $1.6B rate base)
#
# These are ILLUSTRATIVE estimates derived from utility annual reports and
# rate case evidence, applied only to the ELECTRIC rate base.
#
# KNOWN LIMITATIONS (see Part Six of companion article):
# (1) Assessed value ≠ rate base: NY franchise assessment uses income
#     capitalisation and stock-and-debt methods, not original cost.
# (2) NWS avoidance prevents future assets from being built; it does NOT
#     immediately reduce assessed value of existing assets. The property
#     tax saving materialises slowly over years as the avoided asset base
#     compounds, not immediately as the model implies.
# (3) NY franchise tax has both property AND income components. As utility
#     earnings rise under shared savings, the income component increases,
#     partially offsetting the property component reduction.
# (4) Municipal fiscal interests: Con Edison is one of NYC's largest
#     property taxpayers. Rate reform affecting assessed values will face
#     local government scrutiny not captured here.
#
# BOTTOM LINE: The property tax saving shown in results is directionally
# correct but likely overstated in magnitude and timing. It should be
# read as indicative of the category of benefit, not as a precise forecast.
# State-specific analysis should model property tax from actual utility
# tax filings, not from rate base × estimated rate.


# ── DER flex load capture (used in project() for peak reduction) ──────────────
FLEX_LOAD_CAPTURE_BAU    = 0.08   # 8% of load growth captured by flex resources (BAU)
FLEX_LOAD_CAPTURE_SHARED = 0.40   # 40% captured under shared savings (utility manages DER)

print("Section 1 complete — parameters loaded.")
print(f"Utilities: {utilities_to_run}")
print(f"Utility savings share: {UTILITY_SAVINGS_SHARE:.0%}")
print(f"NWS avoidance at maturity: {NWS_AVOIDANCE_OF_ELIGIBLE_MATURE:.0%} of eligible CAPEX")
print(f"Utilisation credit: {FLEX_LOAD_UTILISATION_SHARE_SS:.0%} of load growth uses headroom (SS)")
print(f"Rate freeze: {RATE_FREEZE_YEARS} years (set to 0 to disable)")

## Parameter Guidance

The cell below documents every adjustable parameter, what it controls, its default value, its plausible range, and where to change it. **You do not need to run this cell** — it is documentation. Edit values in Section 1 above, then re-run all cells.

In [ ]:
# ============================================================
# PARAMETER GUIDANCE: HOW TO CHANGE AN ASSUMPTION
# ============================================================
#
# If you disagree with any default value, change it in Section 1 above,
# then re-run all cells (Kernel → Restart & Run All).
#
# This cell provides a structured guide to which parameter to change
# for each type of assumption, and what the plausible range is.
#
# ┌─────────────────────────────────────────────────────────────────────┐
# │ QUESTION                     │ PARAMETER               │ DEFAULT  │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How much of the CAPEX         │ NWS_ELIGIBLE_CAPEX_     │  0.30   │
# │ programme can NWS address?    │ SHARE_START             │         │
# │ (starting share, year 1)      │ Range: 0.20 – 0.50      │         │
# │                               │ Brattle Ontario: 20-40% │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How much of eligible CAPEX    │ NWS_AVOIDANCE_OF_       │  0.80   │
# │ can NWS defer at maturity?    │ ELIGIBLE_MATURE         │         │
# │                               │ Range: 0.50 – 0.90      │         │
# │                               │ Programme average, not  │         │
# │                               │ per-project guarantee   │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How long to reach maturity?   │ NWS_AVOIDANCE_RAMP_YEARS│    9    │
# │                               │ Range: 5 – 15           │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How much do utilities keep    │ UTILITY_SAVINGS_SHARE   │  0.30   │
# │ from shared savings?          │ Range: 0.10 – 0.40      │         │
# │                               │ NY PSC precedent: ~30%  │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How much do NWS contracts     │ OPEX_RATIO              │  0.15   │
# │ cost vs CAPEX avoided?        │ Range: 0.10 – 0.25      │         │
# │                               │ BQDM actual: 12-18%     │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How much of new load uses     │ FLEX_LOAD_UTILISATION_  │  0.15   │
# │ spare capacity (Brattle       │ SHARE_SS                │         │
# │ Untapped Grid mechanism)?     │ Range: 0.00 – 0.30      │         │
# │ Set to 0.0 to disable         │ Brattle base: ~0.33     │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How fast does baseline load   │ BASELINE_LOAD_GROWTH    │  0.005  │
# │ grow per year?                │ Range: 0.003 – 0.010    │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ Is electrification faster or  │ ELECTRIFICATION_RAMP    │ (dict)  │
# │ slower than assumed?          │ Edit the (year,year):   │         │
# │                               │ rate values in the dict │         │
# │ Example — slower scenario:    │                         │         │
# │ (2031,2035): 0.012 not 0.018  │                         │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ What is the starting NYISO    │ CAPACITY_COST_START     │ (dict)  │
# │ capacity cost per MW-year?    │ ConEd: 0.120 $M/MW-yr   │         │
# │                               │ RGE:   0.070 $M/MW-yr   │         │
# │ Source: NYISO ICAP auctions   │ Range: 0.08 – 0.18      │         │
# │ NYC zone 2024; upstate 2024   │                         │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How fast do capacity prices   │ CAPACITY_COST_GROWTH    │  0.03   │
# │ grow in real terms per year?  │ Range: 0.01 – 0.05      │         │
# │ Driven by tightening margins  │ Conservative: 0.02      │         │
# │ as electrification grows      │ Optimistic:   0.05      │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ What is the cost per MW of    │ DIST_UPGRADE_COST_PER_  │ (dict)  │
# │ distribution upgrade for      │ MW = {'ConEd': 4.0,     │         │
# │ each utility?                 │        'RGE':  1.5}     │         │
# │                               │ Urban underground: 3-8  │         │
# │                               │ Rural overhead:  0.5-2  │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ What is the rate freeze       │ RATE_FREEZE_YEARS       │    2    │
# │ duration?                     │ Range: 0 – 5            │         │
# │ Set to 0 to remove freeze     │ NJ precedent: 2 years   │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ What should the benchmark     │ RESIDENTIAL_KWH_MONTH   │  600    │
# │ customer usage be?            │ Range: 400 – 900        │         │
# │                               │ NY PSC standard: 600    │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How quickly does storage      │ STORAGE_GROWTH_SHARED   │  0.22   │
# │ grow under shared savings?    │ STORAGE_GROWTH_BAU      │  0.15   │
# │                               │ Range: 0.10 – 0.35      │         │
# ├─────────────────────────────────────────────────────────────────────┤
# │ How many years ahead does     │ DER_ADOPTION_LEAD_YEARS │    4    │
# │ shared savings accelerate     │ Range: 2 – 8            │         │
# │ DER deployment?               │                         │         │
# └─────────────────────────────────────────────────────────────────────┘
#
# APPLYING A DIFFERENT UTILITY:
# To add a new utility, add its parameters to the UTILITIES dict in
# Section 0, following the same structure as ConEd and RGE.
# Then add it to utilities_to_run = [...] in Section 1.
# Key items needed from a rate case filing:
#   rate_base_start_m, annual_capex_m, total_revenue_m,
#   roe, equity_ratio, electric_customers, peak_demand_mw,
#   annual_energy_gwh, der_storage_mw_start
# And from tariff schedules:
#   delivery_rate_kwh, supply_rate_kwh, basic_charge_month
#
# CONSERVATIVE SCENARIO (sceptical about NWS potential):
#   NWS_ELIGIBLE_CAPEX_SHARE_START = 0.20
#   NWS_AVOIDANCE_OF_ELIGIBLE_MATURE = 0.50
#   NWS_AVOIDANCE_RAMP_YEARS = 15
#   FLEX_LOAD_UTILISATION_SHARE_SS = 0.05
#   OPEX_RATIO = 0.20
#
# OPTIMISTIC SCENARIO (consistent with Brattle base case):
#   NWS_ELIGIBLE_CAPEX_SHARE_START = 0.40
#   NWS_AVOIDANCE_OF_ELIGIBLE_MATURE = 0.90
#   NWS_AVOIDANCE_RAMP_YEARS = 7
#   FLEX_LOAD_UTILISATION_SHARE_SS = 0.25
#   OPEX_RATIO = 0.12

print("Parameter guidance loaded.")
print("This cell contains no executable logic — it is documentation only.")
print("To apply a different assumption, edit Section 1 and re-run all cells.")

## Section 2: Utility Starting Data

Pre-populated from filed rate cases:
- **Con Edison**: Case 25-E-0072, approved January 2026
- **RG&E**: Case 25-E-0379, filed June 2025. PSC set **temporary rates effective June 1, 2026** (+4.0% electric revenue). Final order still pending.

Bills anchored to actual filed tariff rates for a 600 kWh/month customer.  
⚠️ **RG&E status (as of May 2026):** The PSC set temporary rates effective June 1, 2026, granting a 4.0% electric revenue increase — a fraction of the 20.1% total increase originally requested. The final order on Case 25-E-0379 is still pending. The temporary rates were set based on prior approved rate levels, not the filed request, while the Commission continues its review. A proposed settlement has also been filed. All RG&E model parameters remain estimates from the June 2025 filing and should be updated when the final order issues.

In [ ]:
def load_utility(key):
    """Load and derive utility parameters from rate case data."""
    p = UTILITIES[key].copy()
    p.update(SHARED_DEFAULTS)
    p.update(UTILITIES[key])
    p.update(UTILITY_BILL_PARAMS[key])
    p['wacc'] = p['roe'] * p['equity_ratio'] + COST_OF_DEBT * (1 - p['equity_ratio'])
    # Property tax rate applied to ELECTRIC rate base only
    # These are illustrative estimates from utility annual reports.
    # See IMPORTANT NOTE ON PROPERTY TAX in Section 1 for limitations.
    # Using ~1.6% for ConEd and ~1.8% for RGE (electric assets only).
    # Prior versions used 4.6%/5.6% which inadvertently included all
    # utility assets and produced overstated property tax figures.
    p['property_tax_rate'] = 0.016 if key == 'ConEd' else 0.018
    # Tariff-anchored residential bill (600 kWh/month benchmark)
    p['start_delivery_bill_monthly'] = (
        p['delivery_rate_kwh'] * RESIDENTIAL_KWH_MONTH + p['basic_charge_month']
    )
    p['start_supply_bill_monthly'] = p['supply_rate_kwh'] * RESIDENTIAL_KWH_MONTH
    p['start_monthly_bill'] = (
        p['start_delivery_bill_monthly'] + p['start_supply_bill_monthly']
    )
    p['key'] = key   # stored so project() can look up per-utility parameters
    return p

utility_data = {k: load_utility(k) for k in utilities_to_run}

print("Utility starting parameters (from filed rate cases):")
print()
for key, p in utility_data.items():
    print(f"  {UTILITIES[key]['short_name']}")
    print(f"    Rate base:       ${p['rate_base_start_m']:>10,.0f}M")
    print(f"    Annual CAPEX:    ${p['annual_capex_m']:>10,.0f}M")
    print(f"    ROE:             {p['roe']:.1%}   WACC: {p['wacc']:.2%}")
    print(f"    Customers:       {p['electric_customers']:>10,.0f}")
    print(f"    Peak demand:     {p['peak_demand_mw']:>10,.0f} MW")
    print(f"    Starting bill:   ${p['start_monthly_bill']:.0f}/mo  "
          f"(delivery ${p['start_delivery_bill_monthly']:.0f} + supply ${p['start_supply_bill_monthly']:.0f})")
    print()

## Section 3: Model Engine (v11)

The `project()` function runs the full 2026–2050 projection.

**Three shared savings mechanisms:**
1. **NWS CAPEX avoidance** — eligible CAPEX deferred → smaller rate base → lower delivery charge
2. **Capacity market pass-through savings** — NWS reduces peak demand → lower NYISO ICAP obligation → lower supply charge. Computed in Section 4 as `(peak_bau − peak_ss) × capacity_cost_per_mw_year`, where capacity cost grows at 3%/yr real from a 2026 anchor (NYC: $0.120M/MW-yr; RG&E: $0.070M/MW-yr).
3. **Utilisation credit** — flexible load uses existing spare capacity → fixed costs spread over more kWh → lower cost per kWh

Mechanisms 1 and 3 are computed inside `project()`. Mechanism 2 is applied as a post-processing step in Section 4 after both BAU and SS scenarios have run, so the actual peak demand difference can be used.

**Why this is commercially attractive to utilities:**
Shared savings offers utilities a new and politically durable growth engine. Traditional utility earnings grow by building infrastructure — a strategy increasingly generating rate cases, legislative intervention, and customer hostility as bills rise. Shared savings creates a second growth lever: earnings from finding cheaper solutions. A utility that saves customers $200 million a year through batteries, managed charging, and demand response earns $60 million of that directly — without a rate case, without political conflict, and with every customer and regulator as a commercial ally rather than an adversary. The capabilities required — DER aggregation, real-time grid optimisation, flexible load management — are also the foundational skills for operating a high-renewable, highly electrified grid. This is not a defensive accommodation of change. It is a strategic opportunity to lead it.

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────

def get_elec_ramp(year):
    for (y0, y1), r in ELECTRIFICATION_RAMP.items():
        if y0 <= year <= y1: return r
    return 0.005

def get_nws_eligible_share(cum_elec_load):
    """NWS-eligible CAPEX grows with cumulative electrification load."""
    gap = NWS_ELIGIBLE_CAPEX_SHARE_MAX - NWS_ELIGIBLE_CAPEX_SHARE_START
    return NWS_ELIGIBLE_CAPEX_SHARE_START + gap * min(1.0, cum_elec_load * NWS_ELIGIBLE_CAPEX_LOAD_SENSITIVITY)

def get_nws_avoidance_of_eligible(year):
    """NWS avoidance ramps from START to MATURE over ramp years."""
    if year <= 2026: return NWS_AVOIDANCE_OF_ELIGIBLE_START
    if year >= 2026 + NWS_AVOIDANCE_RAMP_YEARS: return NWS_AVOIDANCE_OF_ELIGIBLE_MATURE
    t = (year - 2026) / NWS_AVOIDANCE_RAMP_YEARS
    return (NWS_AVOIDANCE_OF_ELIGIBLE_START
            + t * (NWS_AVOIDANCE_OF_ELIGIBLE_MATURE - NWS_AVOIDANCE_OF_ELIGIBLE_START))

def get_utilisation_credit(year, scenario):
    """
    Utilisation credit — Brattle Untapped Grid mechanism.
    Flexible load uses existing system headroom without triggering proportional
    CAPEX, spreading fixed costs over more kWh and reducing delivery cost/kWh.
    Ramps with DER maturity: 30% of full credit in 2026, 100% by 2035.
    Zero in BAU: no intentional utilisation optimisation.
    """
    if scenario == 'bau': return FLEX_LOAD_UTILISATION_SHARE_BAU
    t = min(1.0, max(0.0, (year - 2026) / NWS_AVOIDANCE_RAMP_YEARS))
    return FLEX_LOAD_UTILISATION_SHARE_SS * (0.30 + 0.70 * t)

def get_storage_mw(params, year, scenario, peak_mw):
    """Battery storage grows logistically toward STORAGE_MAX_PCT_OF_PEAK."""
    base = params['der_storage_mw_start']
    cap  = peak_mw * STORAGE_MAX_PCT_OF_PEAK
    rate = STORAGE_GROWTH_SHARED if scenario == 'shared' else STORAGE_GROWTH_BAU
    eff_yrs = max(0, year - 2026 + (DER_ADOPTION_LEAD_YEARS if scenario == 'shared' else 0))
    k = (cap - base) / base if base > 0 else 0
    if k <= 0: return min(base, cap)
    return min(cap / (1 + k * np.exp(-rate * eff_yrs)), cap)

def get_territory_ev(params, year, scenario):
    """Territory EV count apportioned by customer share; managed fraction applied."""
    share = params['electric_customers'] / 8_000_000
    c2 = SHARED_DEFAULTS['ev_adoption_curve']
    ys2 = sorted(c2); vs = [c2[y] for y in ys2]
    base_ev = int(np.interp(2026, ys2, vs)) * share
    eff_year = min(2050, 2026 + (year - 2026) * 1.25) if scenario == 'shared' else year
    total_ev = max(int(np.interp(eff_year, ys2, vs)) * share, base_ev)
    mev = params['ev_managed_charging_pct_bau'] if scenario == 'bau' else EV_MANAGED_CHARGING_SHARED
    return total_ev, total_ev * mev * SHARED_DEFAULTS['ev_peak_kw'] / 1000

def vlines(ax):
    """Add CLCPA milestone vertical lines to a chart axis."""
    lo, hi = ax.get_ylim()
    for yr, lbl in MY.items():
        ax.axvline(yr, color='gray', ls=':', alpha=0.5, lw=1.2)
        ax.text(yr + 0.25, lo + (hi - lo) * 0.03, lbl, fontsize=7, color='#555', va='bottom')


# ── Core projection function ───────────────────────────────────────────────────

def project(params, scenario='bau', apply_freeze=False):
    """
    Project utility financials and residential customer bills 2026–2050.

    Parameters
    ----------
    params       : dict from load_utility()
    scenario     : 'bau' or 'shared'
    apply_freeze : bool — apply two-year rate freeze overlay

    Returns
    -------
    pd.DataFrame, one row per year, all financial metrics and bill components
    """
    # Resolve per-utility upgrade cost (supports dict or scalar)
    _cost_per_mw = (DIST_UPGRADE_COST_PER_MW.get(params['key'], 2.0)
                    if isinstance(DIST_UPGRADE_COST_PER_MW, dict)
                    else DIST_UPGRADE_COST_PER_MW)
    rows = []
    rb   = params['rate_base_start_m']
    rr   = params['total_revenue_m']
    cust = params['electric_customers']
    egwh = params['annual_energy_gwh']
    pk   = params['peak_demand_mw']
    capex0   = params['annual_capex_m']
    ptbase   = rr * (1 - params['delivery_share_rr'])   # supply/pass-through base
    prop_tax_base = rb * params['property_tax_rate']
    other_om_base = max(0, rr * params['delivery_share_rr'] * 0.35 - prop_tax_base)

    cum_load = 1.0; cum_elec_load = 0.0; deferred_pool = 0.0
    freeze_yr_end  = 2026 + RATE_FREEZE_YEARS
    catchup_yr_end = freeze_yr_end + RATE_FREEZE_RECOVERY_YEARS

    # Bill components (tariff-anchored, tracked separately)
    delivery_bill = params['start_delivery_bill_monthly']
    supply_bill   = params['start_supply_bill_monthly']
    prev_delivery_rr = None
    prev_capex_spent = None

    for year in YEARS:
        elr = get_elec_ramp(year)
        dcr = params['dc_load_growth_annual']
        tlg = BASELINE_LOAD_GROWTH + elr + dcr

        cum_elec_load += (elr + dcr)
        annual_capex   = capex0 * cum_load
        eligible_share = get_nws_eligible_share(cum_elec_load)
        eligible_capex = annual_capex * eligible_share

        smw = get_storage_mw(params, year, scenario, pk)
        _, ev_nws_mw = get_territory_ev(params, year, scenario)
        flc = FLEX_LOAD_CAPTURE_BAU if scenario == 'bau' else FLEX_LOAD_CAPTURE_SHARED

        # NWS physical capability (Option C: MW × cost/MW cross-check)
        st_off  = smw * STORAGE_PEAK_OFFSET_KW_PER_MWH
        dc_fl   = pk * dcr * params['dc_distribution_share'] * params['dc_flexibility_share']
        nws_mw  = (ev_nws_mw + st_off + dc_fl) * NWS_DISPATCH_EFFICIENCY
        nws_val = nws_mw * _cost_per_mw

        # ── Mechanism 1: NWS CAPEX avoidance ─────────────────────────────────
        if scenario == 'bau':
            capex_spent = annual_capex
            nws_opex = av = ea = 0.0
            floor_binding = False
        else:
            aoe = get_nws_avoidance_of_eligible(year)
            # Avoided CAPEX: min of policy ceiling and physical NWS capability
            av = min(min(eligible_capex * aoe, nws_val), eligible_capex)
            capex_spent_raw = annual_capex - av
            floor_binding = False
            # Floor: CAPEX spent must grow by at least 0.5%/yr
            # (NWS defers future projects, not current commitments)
            if prev_capex_spent is not None:
                floor = prev_capex_spent * (1 + CAPEX_SPENT_FLOOR_GROWTH)
                if capex_spent_raw < floor:
                    capex_spent_raw = floor
                    av = max(0, annual_capex - capex_spent_raw)
                    floor_binding = True
            capex_spent = capex_spent_raw
            ea = av / max(1, annual_capex)
            nws_opex = av * OPEX_RATIO
        prev_capex_spent = capex_spent

        # ── Revenue requirement ───────────────────────────────────────────────
        rb = rb * 0.965 + capex_spent   # 3.5% depreciation + new investment
        te = rb * params['wacc']
        prop_tax  = rb * params['property_tax_rate'] * (1 + INFLATION_RATE * 0.5) ** (year - 2026)
        other_om  = other_om_base * (1 + INFLATION_RATE) ** (year - 2026)
        delivery_rr = te + prop_tax + other_om + nws_opex

        # ── Supply revenue requirement ────────────────────────────────────────
        # Supply/pass-through grows with load in both scenarios.
        # Pass-through capacity savings are computed EXTERNALLY after both
        # scenarios run (see Section 4), as:
        #   saving = (peak_bau - peak_ss) × capacity_cost_per_mw_year
        # They are then applied to adjust bills and savings pool retrospectively.
        supply_rr = ptbase * (1 + tlg) ** (year - 2026)
        rrc = delivery_rr + supply_rr

        if scenario == 'bau':
            usr = cs = sp = 0.0
        else:
            # Savings pool: NWS CAPEX avoidance net of OPEX only
            # (capacity market pass-through savings added in Section 4)
            sp  = av - nws_opex
            usr = sp * UTILITY_SAVINGS_SHARE
            cs  = sp * (1 - UTILITY_SAVINGS_SHARE)
            rrc -= cs
            delivery_rr -= cs * params['delivery_share_rr']

        # ── Rate freeze overlay ───────────────────────────────────────────────
        freeze_active = catchup_active = False
        freeze_bill_adj = 0.0
        if apply_freeze:
            prev_bill = (rows[-1]['avg_monthly_bill'] if rows
                         else params['start_monthly_bill'])
            natural_bill = delivery_bill + supply_bill
            if year <= 2026 + RATE_FREEZE_YEARS - 1:
                if natural_bill > prev_bill:
                    deferred = natural_bill - prev_bill
                    deferred_pool += deferred
                    freeze_bill_adj = -deferred
                freeze_active = True
            elif year <= catchup_yr_end:
                years_left = catchup_yr_end - year + 1
                catchup_m = deferred_pool / max(1, years_left)
                freeze_bill_adj = catchup_m
                deferred_pool = max(0, deferred_pool - catchup_m)
                catchup_active = True

        # ── Bill calculation: Mechanisms 1, 2, and 3 ─────────────────────────
        # Mechanism 1 tracked: delivery_rr changes → delivery_bill index
        if prev_delivery_rr is not None and prev_delivery_rr > 0:
            drr_ratio = delivery_rr / prev_delivery_rr
        else:
            drr_ratio = 1.0

        # Mechanism 3: utilisation credit
        # Flexible load uses spare capacity → delivery cost grows more slowly
        # than energy sold → existing fixed costs diluted over more kWh
        uc = get_utilisation_credit(year, scenario)
        delivery_bill *= drr_ratio * (1.0 - uc * tlg)

        # Mechanism 2 tracked: supply grows at (tlg - passthrough_savings)
        supply_growth = (1 + tlg)  # same in both scenarios; pass-through applied in Section 4
        supply_bill  *= supply_growth
        prev_delivery_rr = delivery_rr

        monthly_bill = delivery_bill + supply_bill + freeze_bill_adj

        # Update system state
        cum_load *= (1 + tlg); egwh *= (1 + tlg)
        pk = pk * (1 + tlg * (1 - flc)) if scenario == 'shared' else pk * (1 + tlg)
        cust *= 1.005

        rows.append(dict(
            year=year, scenario=scenario, freeze=apply_freeze,
            # CAPEX breakdown
            capex_total_programme_m=annual_capex,
            capex_eligible_m=eligible_capex,
            capex_ineligible_m=annual_capex * (1 - eligible_share),
            capex_spent_m=capex_spent,
            capex_avoided_m=av,
            capex_avoidance_pct_of_total=ea * 100,
            capex_avoidance_pct_of_eligible=(get_nws_avoidance_of_eligible(year) * 100
                                             if scenario == 'shared' else 0.0),
            nws_eligible_share_pct=eligible_share * 100,
            nws_opex_m=nws_opex,
            floor_binding=floor_binding,
            total_grid_spend_m=capex_spent + nws_opex,
            # Revenue requirement
            delivery_rr_m=delivery_rr, supply_rr_m=supply_rr,
            revenue_req_m=rrc, traditional_earnings_m=te,
            property_tax_m=prop_tax, other_om_m=other_om,
            # Utility financials
            utility_shared_revenue_m=usr,
            total_utility_revenue_m=te + usr,
            roe_earnings_m=rb * params['equity_ratio'] * params['roe'],
            customer_savings_m=cs, savings_pool_m=sp,
            rate_base_m=rb,
            # DER stack
            nws_capable_mw=nws_mw, storage_mw=smw,
            ev_count=get_territory_ev(params, year, scenario)[0],
            # Customer bill (three mechanisms combined)
            avg_monthly_bill=monthly_bill,
            avg_annual_bill=monthly_bill * 12,
            delivery_bill_monthly=delivery_bill,
            supply_bill_monthly=supply_bill,
            utilisation_credit_pct=uc * 100,
            freeze_bill_adj_monthly=freeze_bill_adj,
            # System metrics
            customers=cust, energy_gwh=egwh, peak_mw=pk,
            freeze_active=freeze_active, catchup_active=catchup_active,
            cost_per_mwh=(rrc * 1e3) / max(1, egwh),
            peak_per_ratebase=pk / max(1, rb),
        ))
    return pd.DataFrame(rows)

print("Model engine loaded — v12 with three shared savings mechanisms:")
print("  1. NWS CAPEX avoidance   → smaller rate base → lower delivery revenue requirement")
print("  2. Pass-through savings  → lower peak demand → lower NYISO capacity charges")
print("  3. Utilisation credit    → flexible load uses headroom → fixed costs per kWh fall")

## Section 4: Run All Scenarios

In [ ]:
results = {}
for k in utilities_to_run:
    p = utility_data[k]
    results[k] = {}
    for scenario in ['bau', 'shared']:
        for freeze in [False, True]:
            key = f"{scenario}{'_freeze' if freeze else ''}"
            results[k][key] = project(p, scenario, freeze)

# ── Post-processing: apply capacity market pass-through savings ───────────────
# Pass-through saving in year N = (peak_bau - peak_ss) × capacity_cost_year_N
# This is computed here, after both scenarios run, so we can take the actual
# peak demand difference rather than approximating from a fixed rate.
# The saving is split 70/30 between customers and utility (same as NWS savings).
# Customer share reduces supply bills; utility share adds to shared savings income.

for k in utilities_to_run:
    p_util = utility_data[k]
    cap_cost_start = (CAPACITY_COST_START[k]
                      if isinstance(CAPACITY_COST_START, dict)
                      else CAPACITY_COST_START)
    bau_df = results[k]['bau'].copy()
    sh_df  = results[k]['shared'].copy()

    pt_savings_m = []      # $M/yr — total capacity market saving
    pt_customer_m = []     # $M/yr — customer share (70%)
    pt_utility_m  = []     # $M/yr — utility share (30%)
    pt_monthly_bill = []   # $/month — customer bill reduction

    for yr in YEARS:
        b_row = bau_df[bau_df['year'] == yr].iloc[0]
        s_row = sh_df[sh_df['year'] == yr].iloc[0]
        peak_reduction_mw = b_row['peak_mw'] - s_row['peak_mw']
        peak_reduction_mw = max(0.0, peak_reduction_mw)  # floor at zero
        cap_cost = cap_cost_start * (1 + CAPACITY_COST_GROWTH) ** (yr - 2026)
        saving_m = peak_reduction_mw * cap_cost   # $M in that year
        cust_share_m = saving_m * (1 - UTILITY_SAVINGS_SHARE)
        util_share_m = saving_m * UTILITY_SAVINGS_SHARE
        # Convert customer share to monthly bill reduction
        cust_count = s_row['customers']
        monthly_reduction = (cust_share_m * 1e6) / (cust_count * 12)
        pt_savings_m.append(saving_m)
        pt_customer_m.append(cust_share_m)
        pt_utility_m.append(util_share_m)
        pt_monthly_bill.append(monthly_reduction)

    # Apply pass-through savings to shared savings dataframes (not BAU)
    import pandas as pd
    pt_df = pd.DataFrame({
        'year': YEARS,
        'pt_total_savings_m': pt_savings_m,
        'pt_customer_m': pt_customer_m,
        'pt_utility_m': pt_utility_m,
        'pt_monthly_bill_reduction': pt_monthly_bill,
    })

    for key in ['shared', 'shared_freeze']:
        df = results[k][key].copy()
        df = df.merge(pt_df, on='year', how='left')
        # Reduce supply bill by customer pass-through share
        df['supply_bill_monthly'] = df['supply_bill_monthly'] - df['pt_monthly_bill_reduction']
        df['avg_monthly_bill']    = df['avg_monthly_bill']    - df['pt_monthly_bill_reduction']
        df['avg_annual_bill']     = df['avg_monthly_bill'] * 12
        # Add utility pass-through earnings to shared savings income
        df['utility_shared_revenue_m'] = df['utility_shared_revenue_m'] + df['pt_utility_m']
        df['total_utility_revenue_m']  = df['traditional_earnings_m']   + df['utility_shared_revenue_m']
        # Add to savings pool
        df['savings_pool_m'] = df['savings_pool_m'] + df['pt_total_savings_m']
        df['customer_savings_m'] = df['customer_savings_m'] + df['pt_customer_m']
        results[k][key] = df

    # Store pass-through data for reference
    results[k]['_passthrough'] = pt_df

print("All scenarios complete (including capacity market pass-through adjustment).")
print()
for k in utilities_to_run:
    bau = results[k]['bau']; sh = results[k]['shared']
    b50 = bau.iloc[-1]; s50 = sh.iloc[-1]
    sav_mo = b50['avg_monthly_bill'] - s50['avg_monthly_bill']
    sav_yr = sav_mo * 12
    cum = np.sum((bau['avg_annual_bill'].values - sh['avg_annual_bill'].values)
                 * sh['customers'].values) / 1e9
    bau_cum = np.sum(bau['capex_spent_m'].values) / 1000
    ss_cum  = np.sum(sh['total_grid_spend_m'].values) / 1000
    p = utility_data[k]
    print(f"  {UTILITIES[k]['short_name']}")
    print(f"    Start:  ${p['start_monthly_bill']:.0f}/mo")
    print(f"    BAU 2050:  ${b50['avg_monthly_bill']:.0f}/mo")
    print(f"    SS  2050:  ${s50['avg_monthly_bill']:.0f}/mo")
    print(f"    Saving:    ${sav_mo:.0f}/mo  (${sav_yr:,.0f}/yr,  "
          f"{sav_yr/(b50['avg_annual_bill']):.1%})")
    print(f"    Cum savings: ${cum:.1f}B  |  Grid spend: BAU ${bau_cum:.0f}B  SS ${ss_cum:.0f}B")
    print(f"    (Delivery + supply bill savings — wholesale energy/transmission/policy costs not modelled)")
    drops = [yr for yr in YEARS[1:]
             if sh[sh['year']==yr].iloc[0]['capex_spent_m']
             < sh[sh['year']==yr-1].iloc[0]['capex_spent_m'] - 0.1]
    print(f"    CAPEX dips: {'None ✓' if not drops else drops}")
    print()

## Section 5: Customer Bill Results

Four scenarios per utility:
- **BAU** — Business As Usual (no reform, no freeze)
- **Shared Savings** — Reform without freeze
- **BAU + Freeze** — BAU with 2-year rate freeze
- **SS + Freeze** — Reform with 2-year rate freeze

Bills are for a 600 kWh/month residential customer, anchored to filed tariff rates.

⚠️ **Scope note:** These figures include both the delivery charge (distribution infrastructure) and the supply charge (energy and NYISO capacity costs), tracked separately and anchored to filed tariff rates. Shared savings reform reduces delivery costs through infrastructure avoidance and supply costs modestly through lower peak demand. What is not modelled: independent movements in wholesale energy prices, transmission costs, or policy surcharges — these grow with load at their starting trajectory and are not subject to distribution utility reform. The savings shown represent what the distribution utility can influence. Total electricity bills will also be shaped by factors outside this model's scope.

In [ ]:
for k in utilities_to_run:
    bau   = results[k]['bau'];   sh    = results[k]['shared']
    bau_f = results[k]['bau_freeze']; sh_f = results[k]['shared_freeze']
    name  = UTILITIES[k]['short_name']
    p     = utility_data[k]

    ann_sav_mo = bau['avg_monthly_bill'].values - sh['avg_monthly_bill'].values
    ann_sav_yr = bau['avg_annual_bill'].values  - sh['avg_annual_bill'].values
    cum_sav    = np.cumsum(ann_sav_yr * sh['customers'].values / 1e6) / 1000

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(
        f'GridDividend — {name}\n'
        f'Customer Bills: BAU vs. Shared Savings (600 kWh/month customer)\n'
        f'Three mechanisms: NWS avoidance + pass-through savings + utilisation credit',
        fontsize=11)

    # All 4 scenarios — full timeline
    ax = axes[0, 0]
    ax.plot(YEARS, bau['avg_monthly_bill'],   color=C['bau'],          lw=2.5, label='BAU')
    ax.plot(YEARS, sh['avg_monthly_bill'],    color=C['shared'],        lw=2.5, label='Shared Savings')
    ax.plot(YEARS, bau_f['avg_monthly_bill'], color=C['freeze_bau'],   lw=2, ls='--', label='BAU + 2yr Freeze')
    ax.plot(YEARS, sh_f['avg_monthly_bill'],  color=C['freeze_shared'], lw=2, ls='--', label='SS + 2yr Freeze')
    ax.fill_between(YEARS, sh['avg_monthly_bill'], bau['avg_monthly_bill'], alpha=0.10, color=C['shared'])
    vlines(ax)
    ax.set_title('Monthly Residential Bill — All 4 Scenarios')
    ax.set_ylabel('$/month')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=8)

    # Near-term zoom: freeze effect
    ax = axes[0, 1]
    zoom = [y for y in YEARS if y <= 2040]
    filt = lambda df, c: [v for v, y in zip(df[c], YEARS) if y <= 2040]
    ax.plot(zoom, filt(bau,   'avg_monthly_bill'), color=C['bau'],          lw=2.5, label='BAU')
    ax.plot(zoom, filt(sh,    'avg_monthly_bill'), color=C['shared'],        lw=2.5, label='Shared Savings')
    ax.plot(zoom, filt(bau_f, 'avg_monthly_bill'), color=C['freeze_bau'],   lw=2, ls='--', label='BAU + Freeze')
    ax.plot(zoom, filt(sh_f,  'avg_monthly_bill'), color=C['freeze_shared'], lw=2, ls='--', label='SS + Freeze')
    ax.axvspan(2026, 2028, alpha=0.10, color='blue')
    ax.axvspan(2028, 2032, alpha=0.07, color='orange')
    b27  = bau_f[bau_f['year'] == 2027].iloc[0]
    b27n = bau[bau['year']   == 2027].iloc[0]
    relief = b27n['avg_monthly_bill'] - b27['avg_monthly_bill']
    ax.annotate(
        f'Freeze: ${relief:.0f}/mo\nrelief in 2027',
        xy=(2027, b27['avg_monthly_bill']),
        xytext=(2028.3, b27['avg_monthly_bill'] - 22),
        arrowprops=dict(arrowstyle='->', color='navy', lw=1), fontsize=8, color='navy')
    vlines(ax)
    ax.set_title('Near-Term (2026–2040)\nFreeze gives relief; catch-up follows; structural reform lasts')
    ax.set_ylabel('$/month')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend(fontsize=8)

    # Monthly saving bar chart
    ax = axes[1, 0]
    ax.bar(YEARS, ann_sav_mo,
           color=[C['shared'] if s > 0 else C['bau'] for s in ann_sav_mo],
           alpha=0.82, width=0.8)
    vlines(ax)
    ax.set_title('Monthly Saving per Customer: Shared Savings vs. BAU')
    ax.set_ylabel('$/month')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

    # Cumulative savings
    ax = axes[1, 1]
    ax.fill_between(YEARS, 0, cum_sav, alpha=0.25, color=C['shared'])
    ax.plot(YEARS, cum_sav, color=C['shared'], lw=2.5, label='SS vs. BAU')
    ann_sav_f = bau_f['avg_annual_bill'].values - sh_f['avg_annual_bill'].values
    cum_sav_f = np.cumsum(ann_sav_f * sh_f['customers'].values / 1e6) / 1000
    ax.plot(YEARS, cum_sav_f, color=C['freeze_shared'], lw=2, ls='--', label='SS+Freeze vs. BAU+Freeze')
    vlines(ax)
    ax.set_title('Cumulative Customer Savings ($B, nominal)')
    ax.set_ylabel('$B')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}B'))
    final = cum_sav[-1]
    ax.annotate(f'${final:.1f}B by 2050', xy=(2049, final), xytext=(2037, final * 0.6),
        arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
        fontsize=10, fontweight='bold', color=C['shared'])
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'charts/chart1_bills_{k.lower()}.png', bbox_inches='tight', dpi=130)
    plt.show()

    # Results table
    print(f"\n{name} — Bill Results Table (600 kWh/month)")
    print(f"  {'Year':>4}  {'BAU$/mo':>9}  {'SS$/mo':>8}  {'BAU+Frz':>9}  {'SS+Frz':>8}"
          f"  {'Saving/yr':>10}  {'%':>6}  {'Cum$B':>7}")
    print("  " + "-" * 70)
    for yr in [2026, 2027, 2028, 2030, 2032, 2035, 2040, 2045, 2050]:
        br  = bau[bau['year']   == yr].iloc[0]
        sr  = sh[sh['year']    == yr].iloc[0]
        brf = bau_f[bau_f['year'] == yr].iloc[0]
        srf = sh_f[sh_f['year']  == yr].iloc[0]
        sv  = br['avg_annual_bill'] - sr['avg_annual_bill']
        idx = YEARS.index(yr)
        cumv = np.sum(
            (bau['avg_annual_bill'].values[:idx+1] - sh['avg_annual_bill'].values[:idx+1])
            * sh['customers'].values[:idx+1]) / 1e9
        print(f"  {yr:4d}   {br['avg_monthly_bill']:>7,.1f}   {sr['avg_monthly_bill']:>6,.1f}"
              f"   {brf['avg_monthly_bill']:>7,.1f}   {srf['avg_monthly_bill']:>6,.1f}"
              f"   {sv:>8,.0f}   {sv/br['avg_annual_bill']:>5.1%}   {cumv:>6.1f}")
    print()

## Section 6: Infrastructure Spending — CAPEX & OPEX

Shows utilities still invest heavily under shared savings — NWS defers a growing but bounded share of growth-driven upgrades. Both CAPEX spent and CAPEX avoided increase year-on-year.

Both utilities are shown. **Con Edison**: cumulative grid spend falls from ~$155B BAU to ~$120B SS ($35B, 22% saving). **RG&E**: cumulative grid spend falls from ~$9.0B BAU to ~$6.6B SS ($2.5B, 27% saving). RG&E's proportionally larger saving reflects its programme being more heavily weighted toward growth-driven overhead reinforcements well-suited to NWS deferral.

In [ ]:
for k in utilities_to_run:
    bau  = results[k]['bau']; sh = results[k]['shared']
    name = UTILITIES[k]['short_name']
    bau_cum    = np.cumsum(bau['capex_spent_m'].values)
    ss_cum     = np.cumsum(sh['capex_spent_m'].values)
    ss_nws_cum = np.cumsum(sh['nws_opex_m'].values)

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.suptitle(f'GridDividend — {name}\nAnnual Infrastructure Spending: CAPEX & OPEX', fontsize=12)

    ax = axes[0, 0]
    ax.stackplot(YEARS, sh['capex_ineligible_m'],
                 sh['capex_spent_m'] - sh['capex_ineligible_m'],
                 labels=['Not NWS-eligible (replacement, safety, compliance)',
                         'NWS-eligible CAPEX spent (after avoidance)'],
                 colors=[C['ineligible'], C['capex_spent']], alpha=0.75)
    ax.fill_between(YEARS, sh['capex_spent_m'], sh['capex_total_programme_m'],
                    alpha=0.60, color=C['capex_avoided'], label='CAPEX avoided by NWS')
    ax.plot(YEARS, bau['capex_spent_m'], color=C['bau'], lw=2, ls='--', alpha=0.7,
            label='BAU total CAPEX (ref.)')
    vlines(ax)
    ax.set_title('CAPEX by Category: NWS-Eligible vs. Not Eligible\n'
                 '(Utilities still invest heavily — NWS defers growth-driven upgrades only)')
    ax.set_ylabel('$M/year')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=7, loc='upper left')

    ax = axes[0, 1]
    ax2 = ax.twinx()
    ax.fill_between(YEARS, 0, sh['capex_avoided_m'], alpha=0.40, color=C['capex_avoided'],
                    label='CAPEX avoided ($M)')
    ax.fill_between(YEARS, 0, sh['nws_opex_m'], alpha=0.80, color=C['opex_nws'],
                    label='NWS OPEX ($M)')
    ax.plot(YEARS, sh['capex_avoided_m'], color=C['capex_avoided'], lw=2)
    ax.plot(YEARS, sh['nws_opex_m'],      color=C['opex_nws'],      lw=2)
    ax2.plot(YEARS, sh['nws_eligible_share_pct'],       color='purple', lw=2, ls='--',
             label='NWS-eligible share % (right)')
    ax2.plot(YEARS, sh['capex_avoidance_pct_of_total'], color='black',  lw=2, ls=':',
             label='Eff. avoidance of total % (right)')
    ax2.plot(YEARS, [get_utilisation_credit(y, 'shared') * 100 for y in YEARS],
             color='teal', lw=1.5, ls='-.', label='Utilisation credit % (right)')
    ax2.set_ylim(0, 60); ax2.set_ylabel('%')
    vlines(ax)
    ax.set_title('CAPEX Avoided, NWS OPEX, and Utilisation Credit\n'
                 '(All three grow continuously — no flatline)')
    ax.set_ylabel('$M/year')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    l1, lb1 = ax.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, lb1 + lb2, fontsize=6, loc='upper left')

    ax = axes[1, 0]
    ax.plot(YEARS, bau['capex_spent_m'],    color=C['bau'],    lw=2.5, label='BAU: CAPEX')
    ax.plot(YEARS, sh['total_grid_spend_m'], color=C['shared'], lw=2.5,
            label='SS: CAPEX spent + NWS OPEX')
    ax.fill_between(YEARS, sh['total_grid_spend_m'], bau['capex_spent_m'],
                    alpha=0.13, color=C['neutral'], label='Annual grid spend saving')
    vlines(ax)
    ax.set_title('Total Annual Grid Spend: BAU vs. Shared Savings')
    ax.set_ylabel('$M/year')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=9)

    ax = axes[1, 1]
    ax.fill_between(YEARS, 0, bau_cum / 1000, alpha=0.18, color=C['bau'],
                    label='BAU cumulative CAPEX ($B)')
    ax.fill_between(YEARS, 0, ss_cum / 1000, alpha=0.22, color=C['capex_spent'],
                    label='SS cumulative CAPEX spent ($B)')
    ax.fill_between(YEARS, ss_cum / 1000, (ss_cum + ss_nws_cum) / 1000, alpha=0.55,
                    color=C['opex_nws'], label='SS cumulative NWS OPEX ($B)')
    ax.plot(YEARS, bau_cum / 1000, color=C['bau'], lw=2.5)
    ax.plot(YEARS, (ss_cum + ss_nws_cum) / 1000, color=C['shared'], lw=2.5,
            label='SS total grid spend ($B)')
    b2050 = bau_cum[-1] / 1000; s2050 = (ss_cum[-1] + ss_nws_cum[-1]) / 1000
    ax.annotate(f'BAU: ${b2050:.0f}B', xy=(2050, b2050), xytext=(2040, b2050 * 0.82),
        arrowprops=dict(arrowstyle='->', color=C['bau'], lw=1),
        fontsize=9, color=C['bau'], fontweight='bold')
    ax.annotate(f'SS: ${s2050:.0f}B\n(${b2050-s2050:.0f}B saved)',
        xy=(2050, s2050), xytext=(2040, s2050 * 0.60),
        arrowprops=dict(arrowstyle='->', color=C['shared'], lw=1),
        fontsize=9, color=C['shared'], fontweight='bold')
    vlines(ax)
    ax.set_title('Cumulative Grid Investment 2026–2050')
    ax.set_ylabel('$B')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}B'))
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(f'charts/chart2_capex_opex_{k.lower()}.png', bbox_inches='tight', dpi=130)
    plt.show()

    print(f"\n{name} — CAPEX & NWS OPEX Table")
    print(f"  {'Year':>4}  {'BAU CAPEX':>11}  {'Elig%':>7}  {'Avoided':>9}  "
          f"{'SS Spent':>10}  {'NWS OPEX':>10}  {'Eff.Avoid%':>11}")
    print("  " + "-" * 72)
    for yr in [2026, 2028, 2030, 2035, 2040, 2045, 2050]:
        br = bau[bau['year'] == yr].iloc[0]; sr = sh[sh['year'] == yr].iloc[0]
        print(f"  {yr:4d}   {br['capex_spent_m']:>9,.0f}  "
              f"{sr['nws_eligible_share_pct']:>6.1f}%  "
              f"{sr['capex_avoided_m']:>7,.0f}   "
              f"{sr['capex_spent_m']:>8,.0f}   "
              f"{sr['nws_opex_m']:>8,.0f}   "
              f"{sr['capex_avoidance_pct_of_total']:>9.1f}%")
    print()

## Section 7: Utility Financial Results

Under shared savings, utility revenue exceeds BAU in the **early years** of the programme — the period when the utility most needs the commercial incentive to build NWS capabilities. As the avoided rate base accumulates, the compounding effect of foregone traditional earnings creates a crossover around the mid-2030s. This underscores why regulatory design matters: the shared savings percentage, performance corridor, and ROE framework need to be calibrated to keep the utility commercially motivated throughout the programme, not just in its early years.

**The early-year arithmetic:** the utility earns ~25¢ of shared savings income per $1 of deferred CAPEX but loses only ~7¢ in traditional earnings — a net benefit of ~18¢ per deferred dollar. This is what makes the reform commercially viable in its critical early phase.

In [ ]:
for k in utilities_to_run:
    bau  = results[k]['bau']; sh = results[k]['shared']
    name = UTILITIES[k]['short_name']

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(f'GridDividend v9 — {name}\nUtility Financial Position: BAU vs. Shared Savings',
                 fontsize=12)

    ax = axes[0, 0]
    ax.plot(YEARS, bau['total_utility_revenue_m'], color=C['util_bau'], lw=2.5, label='BAU: total revenue')
    ax.plot(YEARS, sh['traditional_earnings_m'], color=C['util_shared'], lw=2, ls='--',
            label='SS: traditional earnings (WACC × rate base)')
    ax.plot(YEARS, sh['total_utility_revenue_m'], color=C['shared'], lw=2.5,
            label='SS: traditional + shared savings income')
    ax.fill_between(YEARS, sh['traditional_earnings_m'], sh['total_utility_revenue_m'],
                    alpha=0.35, color=C['shared'], label='New: shared savings income stream')
    vlines(ax)
    ax.set_title('Utility Revenue Decomposed\n'
                 '(SS > BAU throughout — this is the commercial incentive for reform)')
    ax.set_ylabel('$M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=8, loc='upper left')

    ax = axes[0, 1]
    ax.plot(YEARS, bau['rate_base_m'], color=C['bau'],    lw=2.5, label='BAU Rate Base')
    ax.plot(YEARS, sh['rate_base_m'],  color=C['shared'], lw=2.5, label='SS Rate Base')
    ax.fill_between(YEARS, sh['rate_base_m'], bau['rate_base_m'],
                    alpha=0.13, color=C['neutral'], label='Avoided infrastructure')
    vlines(ax)
    ax.set_title('Electric Rate Base ($M)')
    ax.set_ylabel('$M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=9)

    ax = axes[1, 0]
    ax.stackplot(YEARS, sh['roe_earnings_m'], sh['utility_shared_revenue_m'],
        labels=['Traditional ROE earnings (on lower rate base)', 'New: shared savings income'],
        colors=[C['util_bau'], C['util_shared']], alpha=0.75)
    ax.plot(YEARS, bau['roe_earnings_m'], color=C['bau'], lw=2, ls='--', label='BAU ROE (ref.)')
    vlines(ax)
    ax.set_title('Utility Earnings Decomposition — Shared Savings Scenario')
    ax.set_ylabel('$M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=9)

    ax = axes[1, 1]
    ax.plot(YEARS, bau['property_tax_m'], color=C['bau'],    lw=2.5, label='BAU property tax')
    ax.plot(YEARS, sh['property_tax_m'],  color=C['shared'], lw=2.5, label='SS property tax')
    ax.fill_between(YEARS, sh['property_tax_m'], bau['property_tax_m'],
                    alpha=0.25, color=C['tax'], label='Annual tax saving')
    vlines(ax)
    ax.set_title('Property Tax: BAU vs. Shared Savings ($M)\n'
                 '(Smaller rate base → smaller property tax → compounding customer benefit)')
    ax.set_ylabel('$M')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'charts/chart3_utility_{k.lower()}.png', bbox_inches='tight', dpi=130)
    plt.show()

## Section 8: Sensitivity Analysis

Tests how results change across parameter ranges. Dot = default value.

**Note on parameter correlation:** The analysis below varies one parameter at a time, holding all others constant. In practice many parameters are correlated — high electrification growth tends to increase NWS opportunity but also increases operational complexity and may reduce effective avoidance rates; higher DER penetration is both a driver of flexibility and a source of dispatch uncertainty. A full probabilistic analysis using Monte Carlo methods with correlated parameter draws would produce wider confidence intervals than the ranges shown here. The single-variable sensitivity charts should be read as illustrating which parameters matter most, not as bounding the full range of possible outcomes.

In [ ]:
PARAM_DEFAULTS = {
    'NWS_ELIGIBLE_CAPEX_SHARE_START':   0.30,
    'NWS_AVOIDANCE_OF_ELIGIBLE_MATURE': 0.80,
    'UTILITY_SAVINGS_SHARE':            0.30,
    'FLEX_LOAD_UTILISATION_SHARE_SS':   0.15,
}
SENSITIVITY_RUNS = {
    'NWS_ELIGIBLE_CAPEX_SHARE_START':   [0.20, 0.25, 0.30, 0.40, 0.50],
    'NWS_AVOIDANCE_OF_ELIGIBLE_MATURE': [0.50, 0.65, 0.80, 0.90],
    'UTILITY_SAVINGS_SHARE':            [0.10, 0.20, 0.30, 0.40],
    'FLEX_LOAD_UTILISATION_SHARE_SS':   [0.00, 0.08, 0.15, 0.25],
}
PARAM_LABELS = {
    'NWS_ELIGIBLE_CAPEX_SHARE_START':   'NWS-eligible\nCAPEX share (start)',
    'NWS_AVOIDANCE_OF_ELIGIBLE_MATURE': 'Avoidance of eligible\nCAPEX (mature)',
    'UTILITY_SAVINGS_SHARE':            'Utility savings share %',
    'FLEX_LOAD_UTILISATION_SHARE_SS':   'Utilisation credit\n(Brattle mechanism)',
}

k = utilities_to_run[0]
p = utility_data[k]; bau_df = results[k]['bau']
all_sens = []
for pname, pvals in SENSITIVITY_RUNS.items():
    recs = []
    for val in pvals:
        orig = globals()[pname]; globals()[pname] = val
        s = project(p, 'shared')
        cum_b = np.sum((bau_df['avg_annual_bill'].values - s['avg_annual_bill'].values)
                       * s['customers'].values) / 1e9
        ur = s[s['year'] == 2050]['total_utility_revenue_m'].values[0]
        recs.append({'parameter': pname, 'value': val, 'cum_b': round(cum_b, 2), 'ur': round(ur, 0)})
        globals()[pname] = orig
    all_sens.append(pd.DataFrame(recs))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(
    f'GridDividend v9 — Sensitivity Analysis ({UTILITIES[k]["short_name"]})\n'
    f'Range of outcomes across parameter assumptions. Dot = default value.',
    fontsize=12)

for ci, (metric, title) in enumerate([
    ('cum_b', 'Cumulative Customer Savings ($B)'),
    ('ur',    'Utility Revenue 2050 ($M)')]):
    ax = axes[ci]; ranges = []
    for s in all_sens:
        pname   = s['parameter'].iloc[0]
        default = PARAM_DEFAULTS[pname]
        lo = s[metric].min(); hi = s[metric].max()
        dr = s[abs(s['value'] - default) < 1e-9]
        mid = dr[metric].values[0] if len(dr) else (lo + hi) / 2
        ranges.append({'label': PARAM_LABELS[pname], 'lo': lo, 'hi': hi, 'mid': mid})
    rdf = pd.DataFrame(ranges).sort_values('hi', ascending=True)
    yp  = list(range(len(rdf)))
    ax.barh(yp, rdf['hi'] - rdf['lo'], left=rdf['lo'],
            color=C['util_shared'], alpha=0.72, height=0.5)
    ax.scatter(rdf['mid'], yp, color='black', zorder=5, s=55, label='Default')
    ax.set_yticks(yp); ax.set_yticklabels(rdf['label'], fontsize=9)
    ax.set_title(title)
    if metric == 'cum_b':
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}B'))
    else:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
    ax.legend(['Default value'], fontsize=8)

plt.tight_layout()
plt.savefig('charts/chart4_sensitivity.png', bbox_inches='tight', dpi=130)
plt.show()

## Section 9: Con Edison vs. RG&E — Side-by-Side Comparison

In [ ]:
if len(utilities_to_run) < 2:
    print("Set utilities_to_run = ['ConEd', 'RGE'] in Section 1 to see comparison.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle('GridDividend v9 — Con Edison vs. RG&E Compared\n'
                 '(Solid = Con Edison · Dashed = RG&E)', fontsize=13)
    panels = [
        ('avg_monthly_bill', 'Monthly Residential Bill ($/month)',  '${:,.0f}'),
        ('avg_annual_bill',  'Annual Residential Bill ($/year)',    '${:,.0f}'),
        ('rate_base_m',      'Electric Rate Base ($M)',             '${:,.0f}M'),
        ('property_tax_m',   'Property Tax on Infrastructure ($M)', '${:,.0f}M'),
        ('capex_spent_m',    'Annual CAPEX Spent ($M)',             '${:,.0f}M'),
        ('cost_per_mwh',     'Cost per MWh Delivered ($)',          '${:,.0f}'),
    ]
    for idx, (col, title, fmt) in enumerate(panels):
        ax = axes[idx // 3][idx % 3]
        for key in utilities_to_run:
            bau = results[key]['bau']; sh = results[key]['shared']
            lbl = UTILITIES[key]['short_name']
            ls  = '--' if key == 'RGE' else '-'
            ax.plot(YEARS, bau[col], color=C['bau'],    lw=2, ls=ls, alpha=0.9, label=f'{lbl} BAU')
            ax.plot(YEARS, sh[col],  color=C['shared'], lw=2, ls=ls, alpha=0.9, label=f'{lbl} SS')
        for yr in MY: ax.axvline(yr, color='gray', ls=':', alpha=0.45, lw=1.1)
        ax.set_title(title, fontsize=10)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _, f=fmt: f.format(x)))
        ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig('charts/chart5_comparison.png', bbox_inches='tight', dpi=130)
    plt.show()

## Section 10: Model Caveats

### Primary Scope Statement

GridDividend is a **strategic policy simulation framework** demonstrating the direction and order of magnitude of the shared savings opportunity. It is not a utility planning model, a rate case replica, or a forecast. The following limitations should be understood before interpreting any result.

### The Three Most Important Limitations

**A. The model is spatially aggregate — this is its most fundamental constraint.**

Real distribution systems are constrained *networks*, not pools of interchangeable capacity. A battery in one neighbourhood cannot relieve a thermal overload in another. NWS effectiveness depends critically on whether the DER is located at or near the constrained feeder, whether its dispatch window coincides with the overload interval, and what type of constraint it is — thermal, voltage, fault current, or protection coordination. The model represents NWS as a system-wide percentage of eligible CAPEX, which is a necessary simplification but one that could materially overstate NWS effectiveness in spatially concentrated or constraint-type-specific portions of the CAPEX programme. Individual feeders will see outcomes ranging from zero NWS benefit (fault-current-driven, asset-condition-driven, or geographically isolated upgrades) to near-complete NWS deferral (growth-driven, thermally constrained, demand-coincident feeders with local DER). The system average used in this model does not predict outcomes at the project level. State-level analysis should obtain the utility's CAPEX programme by constraint type and location and apply NWS eligibility and avoidance rates at that level.

**B. The model tracks the full tariff bill but not all bill drivers.**

The bill figures include both delivery (distribution infrastructure) and supply (energy and NYISO capacity costs), and the reform affects both: delivery falls as infrastructure investment is avoided; supply falls modestly as lower peak demand reduces NYISO capacity market obligations. What the model does not project are independent movements in wholesale energy prices, transmission investment costs, or policy surcharges — these grow with load at their starting trajectory and are unaffected by the shared savings reform. Readers should interpret the savings figures as what a distribution utility under shared savings incentives can influence, not as a projection of total electricity bill movements.

**C. The 80% mature avoidance rate is a programme-level average, not a guarantee.**

The `NWS_AVOIDANCE_OF_ELIGIBLE_MATURE` parameter (default 80%) represents what a well-run, mature NWS programme could achieve across a diverse portfolio of eligible upgrades. It is supported by the Brattle Ontario analysis (NWS cost-effective in 95% of feeder scenarios) and UK DSO flexibility market outcomes. However, it assumes geographic diversity — that the programme spans many feeder types, constraint categories, and seasons so that high avoidance on some projects compensates for low avoidance on others. For any individual feeder or project, the achievable avoidance could be anywhere from zero to 100%. The UK reference should also be used carefully: UK feeders, planning criteria, and market design differ materially from US distribution systems, and demonstrated UK outcomes cannot be imported directly. Applying this parameter to a single constrained corridor or a narrow CAPEX category would likely overstate achievable avoidance significantly.

### Additional Caveats

**1. Time value of money.** Cumulative savings are undiscounted nominal sums. Near-term deferrals are worth more in present value than late deferrals (Brattle Ontario). An NPV analysis with a social discount rate would show lower aggregate savings but the same directional story. Priority future work.

**2. DERMS and programme administration costs.** Not separately tracked. Both GridDividend and the Brattle Ontario analysis omit these. Both analyses therefore slightly overstate net NWS benefits. The operational question is addressed in the companion piece on distribution system operations.

**3. Gas operations** not modelled. Both utilities have large gas distribution businesses interacting with electrification in complex ways.

**4. Full revenue requirement disaggregation** not attempted. Pension costs, storm recovery, environmental remediation, and depreciation schedules are simplified.

**5. NYISO market dynamics** represented by a simple pass-through savings rate. Full capacity market modelling is future work, particularly for application to other regions.

**6. CIAC dynamics** not modelled. Under current regulation, utilities have a financial incentive to specify infrastructure solutions funded by ratepayers — which add to rate base and earn a regulated return — over flexible interconnection alternatives. Developer-funded CIAC contributions reduce rate base under FERC accounting but do not change this fundamental bias toward hardware. The model assumes the full CAPEX programme is utility-funded and rate-base-eligible, slightly overstating BAU bills. Both scenarios are therefore slightly mis-stated in the same direction; savings estimates may be modestly overstated. A future version should add a `developer_funded_capex_fraction` parameter.

**7. Individual customer class impacts** estimated from revenue share proxies, not actual tariff structures. Low-income customers not separately tracked.

**8. RG&E data uncertainty.** Case 25-E-0379 was filed June 2025. The PSC set temporary rates effective June 1, 2026 (+4.0% electric). The final order has not yet been issued as of May 2026. All RG&E parameters are estimates from the June 2025 filing.

**9. Regulatory gaming** not modelled. The model assumes honest accounting.

**10. Utilisation credit scope.** The 15% default is conservative relative to Brattle's Untapped Grid base case (~33%), appropriate because GridDividend models distribution scope only.

**11. Parameter correlation and probabilistic uncertainty.** The sensitivity analysis varies parameters one at a time. In practice many are correlated — high electrification growth increases NWS opportunity but also increases operational complexity; higher DER penetration is both a driver of flexibility and a source of dispatch uncertainty. These correlations mean outputs represent one deterministic path through a wide distribution of plausible outcomes. The directional conclusion is robust; the specific 2050 magnitudes are not.

---

## Section 11: Future Work (Part Nine of companion article)

Key extensions prioritised for future versions:

**Model refinement:** Discounted cash flow analysis · Revenue requirement disaggregation · NYISO capacity market modelling · DERMS cost parameter · CIAC/flexible interconnection dynamics

**Usability:** Graphical parameter interface (Jupyter widgets) · Scenario comparison table · Low-income customer tracking · Gas interaction module

**Other regulatory environments:** Multi-state parameterisation · Vertically integrated utility variant · UK/Australian DSO variant

**Removing fat from customer bills:** Energy efficiency as bill reduction (not surcharge) · Demand response programme unbundling · Interconnection cost reduction for DER developers

**Operational frontier:** Companion piece in preparation on DER management for grid reliability — pre-fault positioning, contingency response, post-fault restoration, protection coordination.

**Legislative design:** A well-designed shared savings statute prioritises the utility's commercial position: annual recognition of shared savings income (no rate case needed to capture what was earned); ring-fencing of that income from rate case compression; and streamlined contract approval to reduce administrative burden on NWS programmes. These protections make the reform commercially attractive to utilities, not just to customers.

---

## References and Further Reading

- Currie, R.A.F. (2025). *Electric Utility Reform in an Age of Electrification.* The States Forum. https://www.statesforum.org/electric-utility-reform-in-an-age-of-electrification/
- Hledik, R., Lam, L., and Peters, K. (2026). *The Untapped Grid: How Better Utilization of the Power System Can Improve Energy Affordability.* The Brattle Group, prepared for GridLab and Utilize Coalition.
- Sergici, S., Ramakrishnan, A., Graham, K., Hledik, R., Grocott, O., and McAdam, C. (2026). *The Value of Using DERs for Distribution System Services in Ontario.* The Brattle Group, prepared for Clean Energy Canada. https://cleanenergycanada.org/wp-content/uploads/2026/04/Distribution-Value-of-DERs-in-Ontario-Final-1.pdf
- NYPSC Case 25-E-0072: Con Edison Electric Rate Case (approved January 2026)
- NYPSC Case 25-E-0379: RG&E Electric Rate Case (filed June 2025; PSC set temporary rates June 1, 2026 at +4.0% electric; final order pending as of May 2026)
- NY CLCPA (2019): Climate Leadership and Community Protection Act, Chapter 106 of the Laws of 2019
- IRS Notice 2016-36: Safe harbor for transfers of interconnection equipment to regulated public utilities
- UKPN DSO (2025). *DSO Performance Panel Report 2024/25.* UK Power Networks. https://dso.ukpowernetworks.co.uk/
- Blake Clough Consulting (2025). *DSO Flexibility Markets: Transforming GB's Grid Economics.* https://www.blakeclough.com/dso-flexibility-markets-transforming-gbs-grid-economics/

---
*GridDividend is open source (MIT License). github.com/[your-handle]/GridDividend*  
*The author welcomes corrections, contributions, and extensions. If you improve a parameter, add a utility, or find a better data source, please open a pull request.*